# Lesson 7a: Deep Q-Networks (DQN) - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand Deep Q-Learning** - Neural networks for Q-function approximation
2. **Master Experience Replay** - Breaking temporal correlations
3. **Learn Target Networks** - Stabilizing the learning target
4. **Understand DQN Algorithm** - Complete deep RL control method
5. **Learn DQN Variants** - Double DQN, Dueling DQN, PER
6. **Understand Rainbow DQN** - State-of-the-art combination
7. **Master Implementation Details** - Critical tricks for success

DQN revolutionized RL in 2015 by achieving human-level performance on Atari games.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. From Linear to Deep Q-Learning

### Recap: Linear Q-Learning (Lesson 6)

**Linear approximation**:
$$Q(s, a; \mathbf{w}) = \mathbf{w}_a^T \mathbf{x}(s)$$

where $\mathbf{x}(s)$ are hand-crafted features (tile coding, RBF, etc.)

**Update**:
$$\mathbf{w} \leftarrow \mathbf{w} + \alpha [R + \gamma \max_{a'} Q(s', a'; \mathbf{w}) - Q(s, a; \mathbf{w})] \nabla Q(s, a; \mathbf{w})$$

**Limitations**:
- ❌ Requires manual feature engineering
- ❌ Limited expressiveness
- ❌ Cannot handle high-dimensional inputs (images, etc.)
- ❌ Doesn't scale to complex domains

### Deep Q-Learning

**Key idea**: Replace linear function with neural network!

$$Q(s, a; \theta) = f_{NN}(s, a; \theta)$$

where:
- $f_{NN}$ is neural network
- $\theta$ are network parameters (weights and biases)
- Network learns features automatically!

**Advantages**:
- ✅ Automatic feature learning
- ✅ High expressiveness (universal function approximation)
- ✅ Handles raw pixels, complex inputs
- ✅ Scales to complex domains

### Neural Network Architectures for Q

**Architecture 1**: Separate output per action

```
Input: state s
    ↓
[Neural Network]
    ↓
Output: [Q(s,a₁), Q(s,a₂), ..., Q(s,aₙ)]
```

**Advantage**: Compute Q for all actions in one forward pass

**Architecture 2**: Single output

```
Input: (state s, action a)
    ↓
[Neural Network]
    ↓  
Output: Q(s,a)
```

**Advantage**: Can handle continuous actions

**DQN uses Architecture 1** for efficiency

### The Instability Problem

**Naive approach**: Replace weights $\mathbf{w}$ with neural network parameters $\theta$

$$\theta \leftarrow \theta + \alpha [R + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta)] \nabla_\theta Q(s, a; \theta)$$

**Problem**: THIS DIVERGES! 💥

**Why?**

1. **Deadly Triad** (Lesson 6):
   - Function approximation ✓ (neural network)
   - Bootstrapping ✓ (TD target)
   - Off-policy ✓ (Q-learning)
   → DANGER!

2. **Correlated samples**:
   - Consecutive transitions highly correlated
   - Violates i.i.d. assumption of SGD
   - Network "forgets" old experiences

3. **Moving target**:
   - Target $R + \gamma \max_{a'} Q(s', a'; \theta)$ depends on $\theta$
   - Changes at every update
   - Like chasing a moving target

4. **Feedback loops**:
   - Q-value update affects policy
   - Policy affects visited states
   - Visited states affect Q-value updates
   - Positive feedback → divergence

### Historical Context

**1990s-2000s**: Deep Q-learning considered impossible
- Researchers tried, failed, gave up
- Believed neural networks couldn't work for RL

**2013-2015**: DeepMind breakthrough
- Two key innovations solved instability:
  1. **Experience Replay**
  2. **Target Networks**
- DQN achieved human-level Atari performance
- Nature paper (2015): "Human-level control through deep RL"
- Started deep RL revolution! 🚀

## 2. Experience Replay

### The Correlation Problem

**Online RL** (naive approach):
```
Observe: s₀, a₀, r₁, s₁
Update network with (s₀, a₀, r₁, s₁)
Observe: s₁, a₁, r₂, s₂  ← Highly correlated with previous!
Update network with (s₁, a₁, r₂, s₂)
...
```

**Problems**:
- Consecutive states are similar → updates are similar
- Network overfits to recent experiences
- "Catastrophic forgetting": forgets old knowledge
- Violates SGD assumption (i.i.d. samples)

### Experience Replay Solution

**Key insight**: Store and reuse experiences!

**Replay Buffer** $\mathcal{D}$:
- Circular buffer (fixed size, e.g., 1M transitions)
- Stores transitions: $(s, a, r, s', \text{done})$
- When full, overwrite oldest

**Algorithm**:
```
1. Act in environment: (s, a, r, s')
2. Store in buffer: D ← (s, a, r, s')
3. Sample random minibatch from D
4. Update network on minibatch
```

### Benefits of Experience Replay

**1. Breaks correlations**:
- Random sampling → i.i.d. minibatches
- Meets SGD assumptions
- Stable gradient estimates

**2. Data efficiency**:
- Each transition used multiple times
- Learn from every experience repeatedly
- 10-100× more sample efficient!

**3. Smooths learning**:
- Averages over different behaviors
- Reduces variance
- More stable updates

**4. Enables off-policy learning**:
- Buffer stores data from any policy
- Can learn from old policies
- Can use expert demonstrations

### Implementation Details

**Buffer size**:
- Small (10K): Faster learning, less diverse
- Large (1M): More diverse, slower to fill
- **Atari standard**: 1,000,000 transitions

**Batch size**:
- Too small: Noisy gradients
- Too large: Slow updates
- **Typical**: 32-256 transitions

**Sampling strategy**:
- **Uniform**: Simple, works well
- **Prioritized** (PER): Sample important transitions more (covered later)

### Mathematical Formulation

**Standard Q-learning**: Update on each $(s, a, r, s')$

**DQN with replay**: Update on random minibatch

**Loss function** (for one transition):
$$L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ (r + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta))^2 \right]$$

**Gradient**:
$$\nabla_\theta L(\theta) = \mathbb{E} \left[ (r + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta)) \nabla_\theta Q(s, a; \theta) \right]$$

**Key**: Expectation over replay buffer $\mathcal{D}$, not just current transition

### Limitations

**Memory**: Requires large buffer (RAM)

**Off-policy only**: Can't use with on-policy methods (A3C, PPO use alternatives)

**Stale data**: Old transitions may not reflect current policy

**Despite limitations**: Experience replay is fundamental to modern deep RL!

## 3. Target Networks

### The Moving Target Problem

**TD target** (semi-gradient Q-learning):
$$y = r + \gamma \max_{a'} Q(s', a'; \theta)$$

**Problem**: Target depends on $\theta$, which we're updating!

**Analogy**: Supervised learning
```
Good:    Labels are fixed
         Minimize: (f(x; θ) - y)²
         
Bad:     Labels change with θ!
         Minimize: (f(x; θ) - g(x; θ))²
         Like chasing your own tail
```

**Result**: Oscillations, divergence, instability

### Target Network Solution

**Key idea**: Use separate network for targets!

**Maintain two networks**:
1. **Online network** $Q(s, a; \theta)$: Updated every step
2. **Target network** $Q(s, a; \theta^-)$: Updated rarely

**Target becomes**:
$$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$

where $\theta^-$ is **fixed** for many steps

**Update schedule**:
```
Every step:
  Update online network θ
  
Every C steps (e.g., C=10,000):
  θ⁻ ← θ  (copy online to target)
```

### Why This Works

**Fixed target** (for C steps):
- Target $y$ is constant
- Like supervised learning
- Stable optimization

**Periodic updates**:
- Incorporate new knowledge
- Prevents target from becoming too stale
- Balances stability and progress

**Reduced oscillations**:
- Online network changes smoothly
- Target network lags behind
- Prevents rapid oscillations

### Hard vs Soft Updates

**Hard update** (DQN original):
$$\theta^- \leftarrow \theta \quad \text{every } C \text{ steps}$$

- Sudden change in target
- Simple to implement
- Used in original DQN (C=10,000)

**Soft update** (DDPG, TD3, SAC):
$$\theta^- \leftarrow \tau \theta + (1-\tau) \theta^- \quad \text{every step}$$

where $\tau \ll 1$ (e.g., 0.001)

- Smooth change in target
- More stable
- Used in actor-critic methods (Lessons 8-10)

### Comparison

**Without target network**:
- Update: $\theta \leftarrow \theta - \alpha \nabla (Q(s,a;\theta) - [r + \gamma \max_{a'} Q(s',a';\theta)])^2$
- Target changes every step
- Unstable, diverges

**With target network**:
- Update: $\theta \leftarrow \theta - \alpha \nabla (Q(s,a;\theta) - [r + \gamma \max_{a'} Q(s',a';\theta^-)])^2$
- Target fixed for C steps
- Stable, converges

### Implementation Details

**Update frequency C**:
- Too small: Not much benefit
- Too large: Target becomes stale
- **Atari standard**: C = 10,000 steps
- **Other domains**: C = 1,000 - 100,000

**Initialization**:
- Both networks start with same weights
- $\theta^- = \theta$ initially

**Computational cost**:
- 2× memory (two networks)
- No extra compute (target network read-only except updates)

### Theory

**Theorem** (Mnih et al., 2015):

Target networks reduce variance of gradient estimates, leading to more stable learning.

**Intuition**: Fixed target = fixed objective → gradient descent works

**Without rigorous proof**, but empirically crucial!

## 4. DQN Algorithm

### Complete DQN Algorithm

**Combining experience replay + target networks:**

```
Initialize:
  - Replay buffer D (capacity N)
  - Online Q-network Q(s,a; θ) with random weights
  - Target Q-network Q(s,a; θ⁻) with θ⁻ = θ
  
For episode = 1 to M:
  Initialize state s
  
  For t = 1 to T:
    # Select action
    With probability ε: a = random action
    Otherwise: a = argmax_a Q(s, a; θ)
    
    # Execute action
    Execute a, observe r, s'
    Store (s, a, r, s', done) in D
    
    # Sample minibatch
    Sample random minibatch of B transitions from D
    
    # Compute targets
    For each transition (sⱼ, aⱼ, rⱼ, s'ⱼ, doneⱼ):
      If doneⱼ:
        yⱼ = rⱼ
      Else:
        yⱼ = rⱼ + γ max_a' Q(s'ⱼ, a'; θ⁻)
    
    # Gradient descent step
    Loss = (1/B) Σⱼ (yⱼ - Q(sⱼ, aⱼ; θ))²
    θ ← θ - α ∇_θ Loss
    
    # Update target network
    Every C steps: θ⁻ ← θ
    
    s ← s'
```

### Key Components

**1. ε-Greedy Exploration**:
- Probability ε: Random action
- Probability 1-ε: Greedy action
- Typically: ε anneals from 1.0 to 0.1 over training

**2. Experience Replay**:
- Store all transitions
- Sample random minibatch
- Break correlations

**3. Target Network**:
- Separate network for targets
- Updated every C steps
- Stabilizes learning

**4. Loss Function**:
- Mean squared TD error
- Huber loss often used (robust to outliers)

### Hyperparameters

**Original DQN (Atari)**:
- Buffer size: N = 1,000,000
- Batch size: B = 32
- Target update: C = 10,000 steps
- Learning rate: α = 0.00025
- Discount: γ = 0.99
- Exploration: ε = 1.0 → 0.1 over 1M steps
- Optimizer: RMSprop

### Network Architecture (Atari)

**Input**: 84×84×4 grayscale frames (4 stacked)

```
Conv1: 32 filters, 8×8, stride 4, ReLU
Conv2: 64 filters, 4×4, stride 2, ReLU
Conv3: 64 filters, 3×3, stride 1, ReLU
Flatten
FC1: 512 units, ReLU
FC2: n_actions units (linear)
```

**Output**: Q(s,a) for each action a

**Parameters**: ~3M total

### Training Process

**Phase 1: Filling buffer** (first 50K steps)
- Random actions (ε=1)
- Fill replay buffer
- No learning yet

**Phase 2: Learning** (50K - 10M steps)
- Anneal ε: 1.0 → 0.1
- Sample and train every step
- Update target network every 10K steps

**Phase 3: Evaluation**
- Set ε = 0.05 (mostly greedy)
- Test on multiple episodes
- Report average return

### Computational Requirements

**Original DQN** (2015):
- Training: 10M frames ≈ 38 hours on GPU
- Per game: ~7-10 days
- 49 Atari games: Months on GPU cluster

**Modern** (2025):
- Much faster with better GPUs/TPUs
- But still computationally expensive
- Sample efficiency remains challenge

## 5. DQN Improvements

### Double DQN (DDQN)

**Problem**: DQN overestimates Q-values

**Why?**
- Target: $y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$
- Max operator selects highest (possibly noisy) estimate
- Noise leads to systematic overestimation
- "Maximization bias"

**Solution** (van Hasselt et al., 2015):

Decouple action selection from value estimation:

**DQN**:
$$y = r + \gamma Q(s', \arg\max_{a'} Q(s', a'; \theta^-); \theta^-)$$

**Double DQN**:
$$y = r + \gamma Q(s', \arg\max_{a'} Q(s', a'; \theta); \theta^-)$$

- **Select** action with online network $\theta$
- **Evaluate** action with target network $\theta^-$

**Benefits**:
- ✅ Reduces overestimation
- ✅ More accurate values
- ✅ Better performance
- ✅ Minimal computation overhead

**Implementation**: Change one line!

### Dueling DQN

**Idea**: Separate value and advantage

**Standard Q-network**: $Q(s,a; \theta)$

**Dueling network**:
$$Q(s,a; \theta) = V(s; \theta_V) + A(s,a; \theta_A)$$

where:
- $V(s)$: State value (how good is this state?)
- $A(s,a)$: Advantage (how much better is this action than average?)

**Architecture**:
```
Input: state s
    ↓
[Shared layers]
    ↓
    ↙     ↘
[V stream] [A stream]
    ↓         ↓
   V(s)    [A(s,a₁), A(s,a₂), ...]
    ↓         ↓
    └────┬────┘
         ↓
Q(s,a) = V(s) + (A(s,a) - mean_a A(s,a))
```

**Centering trick**: Subtract mean advantage (for identifiability)
$$Q(s,a) = V(s) + \left(A(s,a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s,a')\right)$$

**Benefits**:
- ✅ Learns which states are valuable (independent of actions)
- ✅ Better gradient flow
- ✅ Faster learning
- ✅ Especially good when many actions have similar values

### Prioritized Experience Replay (PER)

**Problem**: Uniform sampling wastes computation
- Some transitions more important (higher TD error)
- Some transitions already learned well

**Solution** (Schaul et al., 2016):

Sample transitions based on **priority**:

**Priority**:
$$p_i = |\delta_i| + \epsilon$$

where:
- $\delta_i = r + \gamma \max_{a'} Q(s', a') - Q(s,a)$ is TD error
- $\epsilon > 0$ ensures all transitions can be sampled

**Sampling probability**:
$$P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}$$

where $\alpha$ controls prioritization (α=0: uniform, α=1: proportional)

**Importance sampling weights** (to correct bias):
$$w_i = \left(\frac{1}{N \cdot P(i)}\right)^\beta$$

where $\beta$ anneals from 0.4 to 1.0

**Update**:
$$\theta \leftarrow \theta - \alpha \sum_i w_i \delta_i \nabla Q(s_i, a_i; \theta)$$

**Benefits**:
- ✅ 2× faster learning
- ✅ Better sample efficiency
- ✅ Focus on important transitions

**Cost**:
- More complex implementation
- Additional hyperparameters

### Other Improvements

**Multi-step Learning** (n-step DQN):
- Use n-step returns instead of 1-step
- $y = \sum_{i=0}^{n-1} \gamma^i r_{t+i} + \gamma^n \max_{a'} Q(s_{t+n}, a'; \theta^-)$
- Faster propagation of rewards

**Distributional RL** (C51):
- Model full distribution of returns, not just expectation
- $Z(s,a)$ is distribution (not scalar Q)
- More information, better performance

**Noisy Networks**:
- Add learnable noise to network weights
- Automatic exploration
- Replace ε-greedy

### Rainbow DQN

**Idea**: Combine all improvements!

**Rainbow** (Hessel et al., 2017):
1. Double DQN (decouple selection/evaluation)
2. Dueling networks (value + advantage)
3. Prioritized replay (sample important transitions)
4. Multi-step learning (n-step returns)
5. Distributional RL (model distribution)
6. Noisy networks (exploration)

**Performance**: State-of-the-art on Atari (as of 2017)

**Ablation study**: All 6 contribute, but:
- Most important: Prioritized replay, Multi-step
- Least important: Dueling (still helpful)

**Current status** (2025):
- Rainbow still competitive
- More complex methods exist (MuZero, Agent57)
- But Rainbow is sweet spot of performance/complexity

## 6. Implementation Tricks

### Critical Details for Success

**1. Frame Preprocessing** (Atari):
- Grayscale conversion
- Resize to 84×84
- Frame stacking (4 frames)
- Max pooling over 2 frames (removes flickering)

**2. Reward Clipping**:
- Clip rewards to [-1, +1]
- Makes algorithm game-agnostic
- Stabilizes learning
- **Trade-off**: Loses magnitude information

**3. Terminal Handling**:
```python
if done:
    y = reward
else:
    y = reward + gamma * max_a Q(next_state, a)
```

**4. Gradient Clipping**:
- Clip gradients to [-1, 1] or use norm clipping
- Prevents exploding gradients
- Essential for stability

**5. Huber Loss** (instead of MSE):
$$L_\delta(x) = \begin{cases}
\frac{1}{2}x^2 & \text{if } |x| \leq \delta \\
\delta(|x| - \frac{1}{2}\delta) & \text{otherwise}
\end{cases}$$

- Quadratic near 0 (like MSE)
- Linear for large errors (robust to outliers)
- δ = 1 typical

**6. Learning Rate Schedule**:
- Start: α = 0.00025 (Atari)
- Often fixed (no decay)
- RMSprop or Adam optimizer

**7. Exploration Schedule**:
```python
eps_start = 1.0
eps_end = 0.1 (or 0.01)
eps_decay = 1,000,000 steps

eps = eps_end + (eps_start - eps_end) * exp(-step / eps_decay)
```

**8. Warm-up Period**:
- Fill buffer with 50K random transitions before training
- Ensures diverse initial data

**9. Evaluation Protocol**:
- Separate evaluation (ε=0.05, no learning)
- Run 30 episodes, report average
- Compare to human baseline

### Common Pitfalls

**❌ Forget to update target network**
- Will diverge!
- Must copy weights every C steps

**❌ Sample before buffer is full**
- Need diverse data
- Start learning after 50K transitions

**❌ Wrong gradient calculation**
- Don't backprop through target
- Use `target.detach()` in PyTorch

**❌ Incorrect terminal handling**
- Terminal states: y = r (no bootstrap)
- Common bug: bootstrapping at terminal

**❌ Batch normalization**
- Don't use BatchNorm in DQN!
- Breaks with replay buffer
- Use LayerNorm if needed

**❌ Learning rate too high**
- Will diverge or oscillate
- Start with 1e-4 to 1e-3

### Debugging Tips

**Monitor**:
1. Average reward (should increase)
2. Q-values (should increase, but not explode)
3. TD error (should decrease)
4. Loss (should decrease initially)
5. Gradient norms (should stay reasonable)

**If diverging**:
- Decrease learning rate
- Increase target update frequency
- Add gradient clipping
- Check terminal handling

**If not learning**:
- Check exploration (ε too low?)
- Check network architecture
- Increase learning rate
- Check reward scale

### Summary

**Essential**:
- Experience replay
- Target networks
- Proper terminal handling
- Gradient clipping

**Important**:
- Huber loss
- Reward clipping
- Exploration annealing
- Warm-up period

**Nice-to-have**:
- Frame preprocessing (for Atari)
- Detailed logging
- Checkpoint saving

---

**Lesson 7a Complete!** 🎉

You now understand the theory of Deep Q-Networks, from basic concepts through state-of-the-art improvements. **Lesson 7b** will implement DQN from scratch with all critical components.